# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [23]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [5]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X['timestamp'] = pd.to_datetime(X['timestamp'])
        
        X['hour'] = X['timestamp'].dt.hour
        X['weekday'] = X['timestamp'].dt.weekday
        
        X = X.drop(columns=['timestamp'])
        return X

In [9]:
fe = FeatureExtractor()
df_fe = fe.fit_transform(df)

print(df_fe.head())
print(df_fe.columns)

      uid   labname  numTrials  hour  weekday
0  user_4  project1          1     5        4
1  user_4  project1          2     5        4
2  user_4  project1          3     5        4
3  user_4  project1          4     5        4
4  user_4  project1          5     5        4
Index(['uid', 'labname', 'numTrials', 'hour', 'weekday'], dtype='object')


In [11]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_col):
        self.target_col = target_col
        
    def fit(self, X, y=None):
        self.cat_cols = X.select_dtypes(include='object').columns
        self.cat_cols = [c for c in self.cat_cols if c != self.target_col]
        return self
    
    def transform(self, X):
        X = X.copy()
        
        X = pd.get_dummies(X, columns=self.cat_cols)
        
        y = X[self.target_col]
        X = X.drop(columns=[self.target_col])
        
        return X, y

In [16]:
encoder = MyOneHotEncoder(target_col='weekday')

X, y = encoder.fit_transform(df_fe)

print(X.head())
print(y.head())
print(X.shape, y.shape)

   numTrials  hour  uid_user_0  uid_user_1  uid_user_10  uid_user_11  \
0          1     5       False       False        False        False   
1          2     5       False       False        False        False   
2          3     5       False       False        False        False   
3          4     5       False       False        False        False   
4          5     5       False       False        False        False   

   uid_user_12  uid_user_13  uid_user_14  uid_user_15  ...  labname_lab02  \
0        False        False        False        False  ...          False   
1        False        False        False        False  ...          False   
2        False        False        False        False  ...          False   
3        False        False        False        False  ...          False   
4        False        False        False        False  ...          False   

   labname_lab03  labname_lab03s  labname_lab05s  labname_laba04  \
0          False           False    

In [14]:
class TrainValidationTest:
    def split(self, X, y):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=21, stratify=y
        )
        
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train, y_train, test_size=0.2,
            random_state=21, stratify=y_train
        )
        
        return X_train, X_valid, X_test, y_train, y_valid, y_test

In [17]:
splitter = TrainValidationTest()

X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(X, y)

print(X_train.shape, X_valid.shape, X_test.shape)
print(y_train.shape, y_valid.shape, y_test.shape)

(1078, 43) (270, 43) (338, 43)
(1078,) (270,) (338,)


## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [19]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results = []

    def choose(self, X_train, y_train, X_valid, y_valid):
        best_global_score = 0
        best_model_name = None

        for i, grid in enumerate(self.grids):
            name = self.grid_dict[i]
            print(f"\nEstimator: {name}")

            # tqdm over parameter combinations
            total = len(grid.param_grid[0])
            pbar = tqdm(total=total)

            grid.fit(X_train, y_train)
            pbar.update(total)
            pbar.close()

            best_params = grid.best_params_
            best_train_score = grid.best_score_

            # evaluate on validation
            best_model = grid.best_estimator_
            valid_score = best_model.score(X_valid, y_valid)

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_train_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}")

            self.results.append({
                'model': name,
                'params': best_params,
                'valid_score': valid_score
            })

            if valid_score > best_global_score:
                best_global_score = valid_score
                best_model_name = name

        print(f"\nClassifier with best validation set accuracy: {best_model_name}")
        return best_model_name

    def best_results(self):
        return pd.DataFrame(self.results)

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [22]:
import joblib
from sklearn.metrics import accuracy_score

class Finalize:
    def __init__(self, estimator):
        self.model = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.model.fit(X_train, y_train)
        y_pred = self.model.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy of the final model is {acc}")
        return acc

    def save_model(self, path):
        joblib.dump(self.model, path)
        print("Model was successfully saved.")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [ ]:
df = pd.read_csv('../data/checker_submits.csv')

In [24]:
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('weekday'))
])

In [25]:
# applying pipeline
X, y = preprocessing.fit_transform(df)

In [26]:
splitter = TrainValidationTest()
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(X, y)

In [27]:
# modelSelection setup

svm = SVC()
dt = DecisionTreeClassifier()
rf = RandomForestClassifier()

svm_params = [{
    'kernel': ['rbf'],
    'C': [1, 10],
    'gamma': ['auto'],
    'probability': [True],
    'random_state': [21]
}]

dt_params = [{
    'max_depth': [10, 20],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced'],
    'random_state': [21]
}]

rf_params = [{
    'n_estimators': [50],
    'max_depth': [10, 20],
    'criterion': ['gini', 'entropy'],
    'random_state': [21]
}]

gs_svm = GridSearchCV(svm, svm_params, scoring='accuracy', cv=2)
gs_dt = GridSearchCV(dt, dt_params, scoring='accuracy', cv=2)
gs_rf = GridSearchCV(rf, rf_params, scoring='accuracy', cv=2)

grids = [gs_svm, gs_dt, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

In [28]:
ms = ModelSelection(grids, grid_dict)

best_name = ms.choose(X_train, y_train, X_valid, y_valid)
results_df = ms.best_results()

print(results_df)


Estimator: SVM


100%|██████████| 5/5 [00:01<00:00,  3.56it/s]


Best params: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878

Estimator: Decision Tree


100%|██████████| 4/4 [00:00<00:00, 30.50it/s]


Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 20, 'random_state': 21}
Best training accuracy: 0.799
Validation set accuracy score for best params: 0.856

Estimator: Random Forest


100%|██████████| 4/4 [00:01<00:00,  3.13it/s]

Best params: {'criterion': 'entropy', 'max_depth': 20, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.853
Validation set accuracy score for best params: 0.893

Classifier with best validation set accuracy: Random Forest
           model                                             params  \
0            SVM  {'C': 10, 'gamma': 'auto', 'kernel': 'rbf', 'p...   
1  Decision Tree  {'class_weight': 'balanced', 'criterion': 'gin...   
2  Random Forest  {'criterion': 'entropy', 'max_depth': 20, 'n_e...   

   valid_score  
0     0.877778  
1     0.855556  
2     0.892593  


In [29]:
# get best estimator
best_grid = grids[list(grid_dict.values()).index(best_name)]
best_model = best_grid.best_estimator_

In [30]:
# 6. Finalize
final = Finalize(best_model)

acc = final.final_score(X_train, y_train, X_test, y_test)

model_name = best_name.replace(" ", "_")
filename = f"{model_name}_{acc:.4f}.sav"

final.save_model(filename)

Accuracy of the final model is 0.8994082840236687
Model was successfully saved.
